In [ ]:
# Localiza la raíz del repositorio subiendo desde donde se ejecute el notebook,
# para no depender de una ruta fija de una máquina concreta.
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
while not ((PROJECT_DIR / "data").exists() and (PROJECT_DIR / "notebooks").exists()):
    PROJECT_DIR = PROJECT_DIR.parent

# 1. Importación

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(f"{PROJECT_DIR}/data/processed/madrid/listings_full_features.csv")
df.shape

(18949, 96)

## 2. Preparación de X e y

### 2.1 Identificadores, objetivo y features

Igual que en Barcelona, se excluyen de `feature_cols` las columnas calculadas directamente a partir de `price` (`price_log`, `price_per_accommodate`, `price_per_min_night`): dejarlas dentro sería fuga de información pura. **Fuga corregida más abajo**: `neighbourhood_price_encoded` se calculó en `02_feature_engineering.ipynb` usando todo el dataset, no solo lo que aquí será train. Se corrige en la sección 3.1bis, justo después del split. Por eso `neighbourhood_cleansed` (la columna cruda) sigue en `df` en este punto.

In [3]:
id_cols = ["id", "host_id", "host_profile_id"]
target_col = "price"
leakage_cols = ["price_log", "price_per_accommodate", "price_per_min_night"]
# neighbourhood_cleansed todavía no es una feature (viene de 02_feature_engineering.ipynb
# pendiente de la corrección de la sección 3.1bis): se excluye de X igual que los
# identificadores, pero se mantiene en df para poder usarla justo después del split.
pending_cols = ["neighbourhood_cleansed"]
feature_cols = [c for c in df.columns if c not in id_cols + [target_col] + leakage_cols + pending_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.shape, y.shape

((18949, 88), (18949,))

### 2.2 Booleanas a 0/1

In [4]:
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
len(bool_cols)

57

### 2.3 Nulos restantes

In [5]:
null_cols = X.columns[X.isnull().any()].tolist()
print(null_cols)

X[null_cols] = X[null_cols].fillna(X[null_cols].median())
X.isnull().sum().sum()

['review_scores_rating', 'listing_age_days', 'days_since_last_review']


np.int64(0)

`X` queda con 88 columnas numéricas sin nulos (frente a 69 en Barcelona, por tener más categorías de distrito y tipo de propiedad), e `y` es `price` sin transformar.

## 3. Train/test split

### 3.1 Dividir

Mismo criterio que en Barcelona: 80/20 estratificado por `room_type`, para que `Shared room` (141 anuncios) y `Hotel room` (29, todavía más escaso que en Barcelona) queden repartidos en la misma proporción en train y test.

In [6]:
room_type_cols = [c for c in df.columns if c.startswith("room_type_")]
room_type_for_stratify = df[room_type_cols].idxmax(axis=1)

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df, test_size=0.2, random_state=42, stratify=room_type_for_stratify
)

X_train.shape, X_test.shape

((15159, 88), (3790, 88))

### 3.1bis Corregir la fuga de `neighbourhood_price_encoded`

Exactamente el mismo procedimiento que en Barcelona: recalcular el smoothing solo con `df_train` y aplicar ese mapa a test, con el precio medio de train como valor de respaldo para un barrio que no apareciera en train.

In [7]:
smoothing = 10
global_mean_price_train = y_train.mean()
neigh_stats_train = df_train.groupby("neighbourhood_cleansed")["price"].agg(["mean", "count"])
smoothed_mean_train = (
    neigh_stats_train["count"] * neigh_stats_train["mean"] + smoothing * global_mean_price_train
) / (neigh_stats_train["count"] + smoothing)

df_train["neighbourhood_price_encoded"] = df_train["neighbourhood_cleansed"].map(smoothed_mean_train)
df_test["neighbourhood_price_encoded"] = (
    df_test["neighbourhood_cleansed"].map(smoothed_mean_train).fillna(global_mean_price_train)
)

# X_train/X_test ya tenían la versión con fuga (calculada en la sección 2.1 antes del
# split): se sincronizan con el valor corregido de df_train/df_test.
X_train["neighbourhood_price_encoded"] = df_train["neighbourhood_price_encoded"]
X_test["neighbourhood_price_encoded"] = df_test["neighbourhood_price_encoded"]

df_train = df_train.drop(columns=["neighbourhood_cleansed"])
df_test = df_test.drop(columns=["neighbourhood_cleansed"])

df_train[["neighbourhood_price_encoded"]].describe()

,neighbourhood_price_encoded
count,15159.000000
mean,161.001632
std,37.680500
min,92.332410
25%,127.643913
50%,164.463268
75%,183.117238
max,263.879073


Ningún barrio de test queda fuera del mapa de train (127 barrios, todos representados en un split 80/20 de más de 15.000 filas), así que no hace falta comprobar huecos como en un dataset más pequeño.

### 3.2 Guardar el split

In [8]:
df_train.to_csv(f"{PROJECT_DIR}/data/processed/madrid/listings_train.csv", index=False)
df_test.to_csv(f"{PROJECT_DIR}/data/processed/madrid/listings_test.csv", index=False)

## 4. Baseline ingenuo

### 4.1 Predecir siempre la media

In [9]:
global_mean = y_train.mean()
pred_mean = np.full(len(y_test), global_mean)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_mean)))
print("MAE:", mean_absolute_error(y_test, pred_mean))
print("R2:", r2_score(y_test, pred_mean))

RMSE: 193.89876721202864
MAE: 88.15973742446164
R2: -0.00010611502599755518


### 4.2 Predecir siempre la mediana

In [10]:
global_median = y_train.median()
pred_median = np.full(len(y_test), global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_median)))
print("MAE:", mean_absolute_error(y_test, pred_median))
print("R2:", r2_score(y_test, pred_median))

RMSE: 197.46042770000022
MAE: 81.58725329815303
R2: -0.03718477644733098


Mismo patrón que en Barcelona: el MAE mejora (82.4 frente a 91.6 con la media) pero el R² empeora (-0.03 frente a ~0.00), porque el R² se mide siempre contra la media, no contra la mediana.

### 4.3 Predecir la mediana según `bedrooms`

In [11]:
train_medians_by_bedrooms = X_train.assign(price=y_train).groupby("bedrooms")["price"].median()

pred_bedrooms = X_test["bedrooms"].map(train_medians_by_bedrooms).fillna(global_median)

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_bedrooms)))
print("MAE:", mean_absolute_error(y_test, pred_bedrooms))
print("R2:", r2_score(y_test, pred_bedrooms))

RMSE: 182.94431343410483
MAE: 72.04587071240105
R2: 0.10970524227676437


**Aquí aparece la diferencia más llamativa con Barcelona.** Mejora sobre los baselines anteriores (RMSE 183 frente a 194-197, MAE 72.0 frente a 81.6-88.2), pero el R² apenas llega a **0.11**, muy por debajo del 0.49 que se conseguía en Barcelona con el mismo baseline. No es que `bedrooms` diga menos en Madrid (en `02_feature_engineering.ipynb` se vio una relación creciente igual de clara, solo visible con Spearman/`log(price)`). Es el mismo problema de siempre en este dataset: el R² en escala de precio en bruto es extremadamente sensible a los pocos anuncios con precio muy alto pero genuino (hasta 22.833€/noche), porque el R² penaliza los errores al cuadrado. Un modelo evaluado en precio en bruto se ve mucho más castigado por esa cola en Madrid que en Barcelona.

## 5. Métricas de evaluación

### 5.1 Función `evaluate`

In [12]:
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"modelo": name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

### 5.2 Tabla comparativa de los baselines

In [13]:
rows = [
    evaluate(y_test, pred_mean, "Media"),
    evaluate(y_test, pred_median, "Mediana"),
    evaluate(y_test, pred_bedrooms, "Mediana por bedrooms"),
]

results = pd.DataFrame(rows).set_index("modelo")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,193.90,88.16,-0.00,89.93
Mediana,197.46,81.59,-0.04,66.16
Mediana por bedrooms,182.94,72.05,0.11,59.58


El MAPE sale igual de alto que en Barcelona (60-94%) y por el mismo motivo: hay anuncios muy baratos (`price` mínimo 3.36€) donde un error de pocos euros ya es un porcentaje enorme. Se sigue prefiriendo RMSE/MAE/R² como referencia principal.

## 6. Baseline real: regresión lineal

### 6.1 Entrenar

In [14]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train, y_train_log)

pred_log = model.predict(X_test)
pred_lr = np.expm1(pred_log)

/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/yagocoll/Documents/Master/Airbnb/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


### 6.2 Evaluar

In [15]:
results.loc["Regresión lineal (log)"] = evaluate(y_test, pred_lr, "Regresión lineal (log)")
results.round(2)

,RMSE,MAE,R2,MAPE
modelo,,,,
Media,193.90,88.16,-0.00,89.93
Mediana,197.46,81.59,-0.04,66.16
Mediana por bedrooms,182.94,72.05,0.11,59.58
Regresión lineal (log),162.17,53.15,0.30,30.35


**R²=0.30, MAE≈53€**, muy por encima del mejor baseline ingenuo (R²=0.11, MAE≈72€), pero también muy por debajo del 0.69 que la misma regresión lineal conseguía en Barcelona. Antes de sacar conclusiones sobre qué tan predecible es el mercado de Madrid, hay que revisar la sección 6.3: hay un problema numérico de fondo que puede estar perjudicando a este modelo en concreto más que a Barcelona.

### 6.3 Un aviso a tener en cuenta

In [16]:
import numpy.linalg as la

la.cond(X_train.values)

np.float64(1.4468284334800437e+19)

El número de condición es **~1.4×10¹⁹**, varios órdenes de magnitud peor que en Barcelona (que ya era motivo de aviso allí) y rozando el límite de precisión de `float64` (~10¹⁶). Tiene sentido: Madrid tiene más columnas de distrito (21) y de tipo de propiedad (20) que Barcelona (10 y 14), y cada uno de esos grupos one-hot suma 1 en cada fila, con cuatro grupos así a la vez (`room_type`, `district`, `property_type`, `host_size`) más el intercepto de la regresión, hay más margen para que columnas casi redundantes entre sí se acumulen. Al llamar a `.predict()` incluso saltan avisos de `overflow`/`divide by zero`, algo que no pasaba en Barcelona. Se comprueba si el resultado final está realmente corrupto:

In [17]:
print("valores infinitos en las predicciones:", np.isinf(pred_log).sum())
print("valores NaN en las predicciones:", np.isnan(pred_log).sum())
print("coeficiente máximo (valor absoluto):", np.abs(model.coef_).max())

valores infinitos en las predicciones: 0
valores NaN en las predicciones: 0
coeficiente máximo (valor absoluto): 1.5890347384939187


Ni un solo `inf`/`NaN` en las predicciones, y el coeficiente más grande en valor absoluto es 1.59, nada parecido a los coeficientes disparados que cabría esperar si la solución estuviera realmente rota. El aviso de `overflow` ocurre en un paso intermedio del cálculo matricial (`scikit-learn` sigue resolviendo con SVD), pero el resultado final es numéricamente válido. Conclusión igual que en Barcelona, con más motivo aquí: **no invalida las métricas**, pero sí impide interpretar los coeficientes uno a uno con este modelo tal cual está planteado. Para eso haría falta Ridge/Lasso, que se deja para `04_model_training.ipynb`.

## 7. Conclusiones

Resumen de los resultados del baseline: qué error se puede esperar como mínimo, qué diferencias reales hay con Barcelona, y qué se espera de un modelo más complejo.

### Resultados

| Modelo | RMSE | MAE | R² | MAPE |
|---|---|---|---|---|
| Media | 193.90 | 88.16 | -0.00 | 89.93 |
| Mediana | 197.46 | 81.59 | -0.04 | 66.16 |
| Mediana por `bedrooms` | 182.94 | 72.05 | 0.11 | 59.58 |
| Regresión lineal (log) | 162.17 | 53.15 | 0.30 | 30.35 |

Cada paso mejora sobre el anterior, igual que en Barcelona, pero **el nivel absoluto de R² es mucho más bajo en todos los escalones** (0.11 frente a 0.49 en el mejor baseline ingenuo, 0.30 frente a 0.69 en la regresión lineal). La razón de fondo es la misma en todos los casos y ya se documentó en `01_eda.ipynb` y `02_feature_engineering.ipynb`: la cola de precios altos genuinos de Madrid (asimetría 38.4 en `price`, frente a un valor bastante más moderado en Barcelona) hace que cualquier métrica calculada sobre el precio en bruto (como el R² aquí) sea mucho más frágil, aunque la relación real entre las variables (tamaño, barrio...) y el precio siga siendo, en escala logarítmica o de rangos, tan fuerte como en Barcelona.

### Problemas encontrados y cómo se trataron

- **Fuga de información directa**: `price_log`, `price_per_accommodate` y `price_per_min_night`, excluidas en la sección 2, igual que en Barcelona.
- **Fuga más leve, ahora corregida**: `neighbourhood_price_encoded` recalculada solo con `df_train` en la sección 3.1bis, igual que en Barcelona.
- **Muestra escasa en `Shared room`/`Hotel room`** (141 y 29 anuncios): el split estratifica por `room_type`, igual que en Barcelona.
- **Multicolinealidad severa, más extrema que en Barcelona**: número de condición ~1.4×10¹⁹, con avisos de `overflow` al predecir. Se comprobó que las predicciones y los coeficientes siguen siendo finitos y razonables: no invalida las métricas, sí impide interpretar los coeficientes de este modelo lineal uno a uno.
- **R² mucho más bajo que en Barcelona en todos los baselines**: no es un error, es que el R² sobre precio en bruto es una métrica frágil cuando hay una cola de precios altos genuinos tan marcada como la de Madrid (ver sección 4.3). No se corrige aquí: el objetivo de `03_model_baseline.ipynb` es fijar el listón con el mismo criterio que Barcelona, no maquillar el número, pero es una señal a vigilar en `04_model_training.ipynb`.

### El listón para `04_model_training.ipynb`

Cualquier modelo más complejo tiene que superar **R²=0.30 / MAE≈53€** para justificar la complejidad añadida, un listón bastante más bajo que el de Barcelona (R²=0.69), lo que en principio deja más margen de mejora para los modelos de árboles. Un modelo de árboles, además, no sufre la multicolinealidad de la sección 6.3 ni necesita la imputación por mediana de la sección 2.3.

### Lo que queda guardado

`listings_train.csv` y `listings_test.csv` en `data/processed/madrid/` (15.159 / 3.790 filas, split 80/20 estratificado por `room_type`, `random_state=42`), con `neighbourhood_price_encoded` ya corregido, para que `04_model_training.ipynb` y `05_model_evaluation.ipynb` trabajen sobre las mismas filas.